<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1igctI71MHK5cMTUSyev8wLLhGNfveeAQG_wisKRhQyE/edit?usp=sharing

In [75]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [76]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1igctI71MHK5cMTUSyev8wLLhGNfveeAQG_wisKRhQyE/edit?usp=sharing')

In [99]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最後面的5筆資料
df.tail()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
4,2026-03-05,9:15,水果,apple,20,0,0,0,0
5,2025-09-11,9:00,早餐三明治,50,,,,,
6,2026-03-05,10:10,食物,午餐,80,YC,師大學餐,現金,
7,2026-03-28,15:26,追星,應援物,60,YC,林口體育館,現金,
8,2026-03-08,13:02,食物,地瓜球+芭樂,100,YC,林口體育館,現金,


In [100]:
import datetime

In [102]:
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY/MM/DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y/%m/%d')

time_str = input("請輸入時間 (格式：HH:MM): ")
# 檢查時間格式是否正確
datetime.datetime.strptime(time_str, '%H:%M')

category = input("請輸入分類: ")
item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))
payer = input("請輸入付款人: ")
location = input("請輸入地點: ")
payment_method = input("請輸入支付方式: ")
notes = input("請輸入備註: ")

請輸入日期 (格式：YYYY/MM/DD): 2026/03/22
請輸入時間 (格式：HH:MM): 14:25
請輸入分類: 追星
請輸入品項: 三星手機
請輸入金額: 500
請輸入付款人: YU
請輸入地點: 台北大巨蛋
請輸入支付方式: 轉帳
請輸入備註: 訂金3000


In [103]:
date_str

'2026/03/22'

In [104]:
time_str

'14:25'

In [105]:
category

'追星'

In [106]:
item

'三星手機'

In [107]:
amount

500.0

In [108]:
payer

'YU'

In [109]:
location

'台北大巨蛋'

In [110]:
payment_method

'轉帳'

In [111]:
notes

'訂金3000'

In [112]:
# 創建一個新的 DataFrame 來代表你要新增的資料
new_data = pd.DataFrame([
    {
        '日期': date_str,
        '時間': time_str,
        '分類': category,
        '品項': item,
        '金額': amount,
        '付款人': payer,
        '地點': location,
        '支付方式': payment_method,
        '備註': notes
    }
])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [113]:
# 刪除 df 中所有欄位都完全相同的重複資料
df_unique = df.drop_duplicates()

# 顯示處理後的 DataFrame 的前幾行，以確認重複資料已移除
display(df_unique)

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2025-09-11,9:30,,早餐三明治,50,,,,
1,2025-09-16,,交通,捷運,20,Pecu,,,
2,2025-09-18,,外食,早餐,80,Pecu,,,
3,2025-09-26,,交通,車費儲值,100,H,,,
4,2026-03-05,9:15,水果,apple,20,0,0,0,0
5,2025-09-11,9:00,早餐三明治,50,,,,,
6,2026-03-05,10:10,食物,午餐,80,YC,師大學餐,現金,
7,2026-03-28,15:26,追星,應援物,60,YC,林口體育館,現金,
8,2026-03-08,13:02,食物,地瓜球+芭樂,100,YC,林口體育館,現金,
9,2026/03/22,14:25,追星,三星手機,500.0,YU,台北大巨蛋,轉帳,訂金3000


In [114]:
import pandas as pd

# 取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 取得現有的所有資料，包括標題
all_values = worksheet.get_all_values()

# 檢查是否有標題列
if all_values and len(all_values) > 0:
    # 保留標題列
    header = all_values[0]
    # 清空所有內容，只保留標題列
    worksheet.clear()
    worksheet.update([header], 'A1') # 寫回標題列
    print("工作表已清空，並保留標題列。")
else:
    print("工作表為空或沒有標題列，無需清空。")

# 將 df_unique 的資料寫回工作表
# 將 DataFrame 轉換為列表的列表，並處理 NaN 值
data_to_write_back = df_unique.fillna('').values.tolist()

# 從 A2 開始寫入資料，因為 A1 已經是標題了
if data_to_write_back:
    worksheet.append_rows(values=data_to_write_back, value_input_option='USER_ENTERED')
    print("df_unique 資料已成功寫回 Google 工作表！")
else:
    print("df_unique 為空，沒有資料寫入工作表。")

工作表已清空，並保留標題列。
df_unique 資料已成功寫回 Google 工作表！


In [115]:
import numpy as np

# 計算總花費
# 將 '金額' 欄位轉換為數值型別，並將無法轉換的值設為 NaN，然後將 NaN 填補為 0
total_amount_spent = pd.to_numeric(df_unique['金額'].astype(str).str.replace(',', ''), errors='coerce').fillna(0).sum()
print(f"總共花費了: {total_amount_spent}")

總共花費了: 1010.0


In [116]:
df_unique_copy = df_unique.copy()
df_unique_copy['金額'] = pd.to_numeric(df_unique_copy['金額'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
category_summary = df_unique_copy.groupby('分類')['金額'].sum().reset_index()
print("各分類總花費:")
print(category_summary)

各分類總花費:
      分類     金額
0          50.0
1     交通  120.0
2     外食   80.0
3  早餐三明治    0.0
4     水果   20.0
5     追星  560.0
6     食物  180.0


In [117]:
summary_data = [category_summary.columns.tolist()] + category_summary.values.tolist()

try:
    summary_worksheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1igctI71MHK5cMTUSyev8wLLhGNfveeAQG_wisKRhQyE/edit?usp=sharing').worksheet('Summary')
    summary_worksheet.clear()
    print("現有的 'Summary' 分頁已清空。")
except gspread.exceptions.WorksheetNotFound:
    summary_worksheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1igctI71MHK5cMTUSyev8wLLhGNfveeAQG_wisKRhQyE/edit?usp=sharing').add_worksheet(title="Summary", rows="100", cols="20")
    print("已創建新的 'Summary' 分頁。")

summary_worksheet.update(values=summary_data, range_name='A1')
print("分類統計資料已成功寫入 'Summary' 分頁。")

現有的 'Summary' 分頁已清空。
分類統計資料已成功寫入 'Summary' 分頁。
